# Hyperbolic Herbarium — Full Training on Google Colab

Trains the hyperbolic projection layer and hierarchical classifier heads on all California CCH2 specimens (517K images, streamed from CCH2 servers). BioCLIP-2 backbone stays frozen.

**Runtime**: T4 GPU (free tier). Estimated time: 4–8h for 15 epochs on 517K specimens.

**Steps**:
1. Install dependencies
2. Mount Google Drive (for checkpoint saving)
3. Clone repo and download CA pipeline data
4. Prepare label encoders
5. Train
6. Download checkpoint

In [ ]:
# @title 1. Install dependencies
!pip install -q open_clip_torch geoopt peft faiss-cpu dwca-reader pyarrow pyyaml tqdm requests pillow pandas

In [ ]:
# @title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/hyperbolic_herbarium'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_DIR}')

In [ ]:
# @title 3. Clone repo
import os

REPO_URL = 'https://github.com/beck-2/herbarium_RAG.git'  # update if repo moves
REPO_DIR = '/content/herbarium_RAG'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
import sys
sys.path.insert(0, REPO_DIR)
print('Repo ready:', os.getcwd())

In [ ]:
# @title 4a. Download all CCH2 collections (skip if already on Drive)
import os

CCH2_DRIVE = f'{DRIVE_DIR}/cch2_data'
CCH2_LOCAL = '/content/data/raw/symbiota/cch2'

# Check if we already have the parquet cached on Drive
PARQUET_DRIVE = f'{DRIVE_DIR}/specimens_encoded.parquet'
LABEL_ENC_DRIVE = f'{DRIVE_DIR}/label_encoders.json'

if os.path.exists(PARQUET_DRIVE) and os.path.exists(LABEL_ENC_DRIVE):
    print('Found cached parquet on Drive — skipping CCH2 download.')
    !mkdir -p data/processed/regions/california
    !cp {PARQUET_DRIVE} data/processed/regions/california/specimens_encoded.parquet
    !cp {LABEL_ENC_DRIVE} data/processed/regions/california/label_encoders.json
    SKIP_DOWNLOAD = True
else:
    print('Downloading CCH2 data (this will take ~30-60 min on Colab) ...')
    SKIP_DOWNLOAD = False

In [ ]:
# @title 4b. Run data pipeline (if not cached)
if not SKIP_DOWNLOAD:
    !bash scripts/download_data.sh
    !python scripts/build_california_pipeline.py --dry-run
    !python scripts/prepare_training_data.py
    
    # Cache to Drive for future runs
    !cp data/processed/regions/california/specimens_encoded.parquet {PARQUET_DRIVE}
    !cp data/processed/regions/california/label_encoders.json {LABEL_ENC_DRIVE}
    print(f'Cached parquet to Drive: {PARQUET_DRIVE}')

In [ ]:
# @title 4c. Build full training manifest (all 517K specimens with image URLs)
import pandas as pd
from pathlib import Path

df = pd.read_parquet('data/processed/regions/california/specimens_encoded.parquet')
print(f'Total specimens with images: {len(df):,}')
print(f'Families: {df["family"].nunique()}, Genera: {df["genus"].nunique()}, Species: {df["scientific_name"].nunique()}')

manifest_path = Path('data/processed/regions/california/train_full.txt')
manifest_path.write_text('\n'.join(df['occurrence_id'].tolist()))
print(f'Wrote manifest: {manifest_path} ({len(df):,} entries)')

In [ ]:
# @title 5. Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')

In [ ]:
# @title 6. Train (auto-resumes from last.pt if present on Drive)
import os

CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints/global'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Sync any existing checkpoint from Drive into the working directory
# (so training can resume after a Colab disconnect)
import shutil
for ckpt_name in ['last.pt', 'best.pt']:
    drive_ckpt = f'{CHECKPOINT_DIR}/{ckpt_name}'
    local_ckpt = f'checkpoints/global/{ckpt_name}'
    os.makedirs('checkpoints/global', exist_ok=True)
    if os.path.exists(drive_ckpt) and not os.path.exists(local_ckpt):
        shutil.copy2(drive_ckpt, local_ckpt)
        print(f'Restored {ckpt_name} from Drive')

# Resume from last.pt if available
resume_arg = []
if os.path.exists('checkpoints/global/last.pt'):
    resume_arg = ['--resume', 'checkpoints/global/last.pt']
    print('Resuming from last.pt')
else:
    print('Starting fresh training run')

cmd = [
    'python', 'src/train/train_global.py',
    '--dataset', 'data/processed/regions/california',
    '--train-manifest', 'data/processed/regions/california/train_full.txt',
    '--epochs', '15',
    '--batch-size', '64',
    '--lr', '2e-4',
    '--warmup-steps', '500',
    '--device', 'cuda',
    '--output', 'checkpoints/global/',
] + resume_arg

print('Running:', ' '.join(cmd))
!{' '.join(cmd)}

# Copy checkpoints back to Drive after training
for ckpt_name in ['last.pt', 'best.pt']:
    local = f'checkpoints/global/{ckpt_name}'
    if os.path.exists(local):
        shutil.copy2(local, f'{CHECKPOINT_DIR}/{ckpt_name}')
        print(f'Saved {ckpt_name} → Drive')

In [ ]:
# @title 7. Verify checkpoint
import torch
ckpt_path = f'{CHECKPOINT_DIR}/best.pt'
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=True)
print(f'Checkpoint saved at epoch {ckpt["epoch"]} with val_loss={ckpt["val_loss"]:.4f}')
print(f'proj keys: {list(ckpt["proj_state"].keys())[:5]} ...')
print(f'heads keys: {list(ckpt["heads_state"].keys())[:5]} ...')

In [ ]:
# @title 8. (Optional) Download checkpoint to local machine
# Run this cell to get a download link for the checkpoint file
from google.colab import files
files.download(f'{CHECKPOINT_DIR}/best.pt')